# Multitask fine-tuning: MedText, PubMedQA and MedMCQA

Second stage of the pipeline. One model is fine-tuned on three medical tasks at once, starting either from the original GPT-2 or from the PubMed-pretrained GPT-2 (`pretrain_pubmed.ipynb`). Comparing the two answers the main question of the project: **does domain pretraining on PubMed help the downstream tasks?**

| Task | What the model learns | Test set | Metric |
|---|---|---|---|
| MedText | write the reference answer to a clinical question | MedText test (142) | answer-only loss |
| PubMedQA | answer yes / no / maybe from an abstract, then explain | official test set (500) | accuracy, macro F1 |
| MedMCQA | pick A-D for an exam question, then explain | official validation set (4,181) | accuracy, macro F1 |

Training and evaluation are tracked in MLflow (experiment `gpt2-med-multitask`).

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.config import load_multitask_config
from src.tracking import configure_tracking

config = load_multitask_config("configs/gpt2_multitask.yaml")
configure_tracking(config)

## 1. Data

`prepare_multitask()` (`src/multitask.py`) turns every dataset into prompt-completion pairs with a shared format, and builds the splits:

- **PubMedQA**: the official 500 test questions; the other 500 labelled questions are split into 450 train and 50 validation, stratified by decision. The artificial set (PQA-A, 211k questions, 93% "yes") is used for training only, balanced by decision.
- **MedMCQA**: 5% of the train split is held out for model selection; the official validation set is the test set (the official test set has no answers). About half of the questions have an explanation, which is added after the answer letter.
- **Mixture**: per epoch, MedText and PubMedQA-labelled repeated 5 times, 15,000 "yes" + 15,000 "no" PQA-A questions, and 60,000 new MedMCQA questions (about 98,000 examples). The training set holds 5 such epochs.
- Examples longer than the 1,024-token context are dropped (a few long MedMCQA and PQA-A questions, and 2 MedMCQA test questions).

In [ ]:
from src.multitask import load_multitask, prepare_multitask

prepare_multitask(config)
stats = json.loads((Path(config.multitask.processed_dir) / "stats.json").read_text())
splits = load_multitask(config.multitask.processed_dir)

pd.Series(stats["rows"], name="examples").to_frame()

In [ ]:
print(f"Examples per mixture epoch: {stats['examples_per_epoch']:,}")
print("Training examples by task (all epochs):", stats["train_tasks"])
for name, counts in stats["test_labels"].items():
    majority = max(counts.values()) / sum(counts.values())
    print(f"{name}: {counts} -> always answering the most frequent label scores {majority:.1%}")

### One example per task

In [ ]:
for task in ("medtext", "pubmedqa", "medmcqa"):
    example = splits[f"dev_{task}"][0]
    print(f"===== {task} =====")
    print(example["prompt"][-600:] + example["completion"][:300])
    print()

## 2. Zero-shot baselines

Before any fine-tuning: the original GPT-2 and the PubMed-pretrained GPT-2 on the three test sets. For PubMedQA and MedMCQA the model scores each possible answer (the log-probability of `yes`/`no`/`maybe` or `A`-`D` right after the prompt) and the most likely one is its prediction; this works even for models that cannot follow the answer format yet.

In [ ]:
from src.multitask_train import evaluate_model
from src.tracking import log_evaluation

pretrained_uri = "models:/gpt2-pubmed-dapt@candidate"
baselines = {}
for name, model_id in [("GPT-2", config.model.model_id), ("GPT-2 + PubMed", pretrained_uri)]:
    baselines[name] = evaluate_model(model_id, config, prefix="test")
    log_evaluation(model_id, "zero_shot", baselines[name], config)

pd.DataFrame(baselines).T.round(4)

## 3. Hyperparameter search (Optuna, run in a terminal)

The learning rate and batch size selected for MedText alone (1,129 examples) are not necessarily good for a mixture of ~98,000 examples per epoch, so they are searched again with Optuna before the full runs:

- **Search space** (`sweep` in the YAML): learning rate `1e-5`-`3e-4` (log scale), batch size 8/16/32, constant or cosine schedule, 0% or 5% warmup; 12 trials with the TPE sampler.
- **Budget**: each trial trains for one mixture epoch and is evaluated 4 times on the combined dev set; the median pruner stops trials that are worse than earlier ones at the same point. The learning-rate schedule has the length of a full run, so a trial's epoch matches the first epoch of the final run.
- **Fairness**: the search always uses the original GPT-2, and both base models are then fine-tuned with the same values.
- **No test data**: trials only see the dev sets.

```bash
nohup uv run python -m src.multitask_train --config configs/gpt2_multitask.yaml --sweep > multitask_sweep.log 2>&1 &
```

Then copy the best values into the YAML: `learning_rate` and `batch_size` under `finetune`, `lr_scheduler_type` and `warmup_ratio` under `training`. Keeping them in the config (and in git) makes the final runs reproducible.

In [ ]:
sweep_rows = json.loads(Path(config.sweep.results_path).read_text())
pd.DataFrame(sweep_rows).sort_values("best_dev_combined_loss").round(6)

## 4. Fine-tuning (run in a terminal)

Each command fine-tunes the three seeds in `finetune.seeds` with the hyperparameters chosen in section 3 and logs one `multitask-<base>` MLflow run with a nested run per seed. Training stops when the combined dev loss has not improved for 2 mixture epochs; the best epoch is scored on the test sets. The best seed is registered as `models:/gpt2-med-multitask@<base>-best`.

From the original GPT-2:

```bash
nohup uv run python -m src.multitask_train --config configs/gpt2_multitask.yaml > multitask_gpt2.log 2>&1 &
```

From the PubMed-pretrained GPT-2 (the `candidate` alias, or `champion` once promoted):

```bash
nohup uv run python -m src.multitask_train --config configs/gpt2_multitask.yaml --model-id models:/gpt2-pubmed-dapt@candidate > multitask_dapt.log 2>&1 &
```

Run them one after the other: both use the GPU.

## 5. Results

Mean and standard deviation over the three seeds, read from the `multitask-<base>` runs in MLflow. A difference between the two base models matters only if it is clearly larger than the standard deviations.

In [ ]:
import mlflow

experiment = mlflow.get_experiment_by_name(config.mlflow.experiment_name)
summary_runs = mlflow.search_runs(
    [experiment.experiment_id],
    filter_string="tags.task = 'multitask' AND tags.stage = 'finetune'",
    order_by=["start_time DESC"],
    output_format="list",
)
# The summary runs are the parents: they have no seed tag and hold the *_mean metrics
summary_runs = [run for run in summary_runs if "finetune_seed" not in run.data.tags]

rows = {}
for run in summary_runs:
    base = run.data.tags["base_model"]
    if base in rows:
        continue  # keep the latest run per base model
    metrics = run.data.metrics
    rows[base] = {
        name: f"{metrics[f'{name}_mean']:.4f} ± {metrics[f'{name}_std']:.4f}"
        for name in (
            "test_medtext_loss",
            "test_pubmedqa_accuracy",
            "test_pubmedqa_macro_f1",
            "test_medmcqa_accuracy",
            "test_medmcqa_macro_f1",
        )
        if f"{name}_mean" in metrics
    }

pd.DataFrame(rows).T